# AlphaTrain Pillar 2S: Topological Mastery

**Teacher:** 2R epoch 3 (MCTS mean=2,389 @400 sims, v6 mean=6,971).
**Data:** V6 (1,654 games, 5.4M states, temperature_moves=5).

Key changes from 2R:
- **max_score=200** (was 100) — removes ceiling clamp, 0% clipped values
- Stronger teacher data (mean 6,971 vs 5,117)
- Cleaner openings (temperature_moves=5 vs 15)
- val_weight=0.1, rank_weight=1.0, hard ranking pairs (top-1 vs top-5)
- 10 epochs, eval bracket at 3/6/9

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v6_density_g098_ms200.pt.gz` — V6 tensor (411MB)
3. `pillar2r_epoch_3.pt` — base model (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying model...')
shutil.copy(f'{DRIVE}/pillar2r_epoch_3.pt', '/content/alphatrain/data/model.pt')
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress V6 density reward data
print('Decompressing V6 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v6_density_g098_ms200.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

# Sanity check distribution
import torch
d = torch.load('/content/alphatrain/data/selfplay.pt', weights_only=True, map_location='cpu')
bins = torch.linspace(0, float(d['max_score']), int(d['num_value_bins']))
vals = (d['val_targets'] * bins).sum(dim=-1)
iqr = torch.quantile(vals.float(), 0.75) - torch.quantile(vals.float(), 0.25)
print(f'\nmax_score={d["max_score"]}, bins={d["num_value_bins"]}')
print(f'Value IQR: {iqr:.1f} (must be >2.0)')
assert iqr > 2.0, f'Value blob NOT fixed! IQR={iqr:.2f}'
print(f'Clipped (V>={d["max_score"]}): {(vals >= float(d["max_score"]) - 1).sum()} (must be 0)')
del d, vals

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2S: Topological Mastery — V6 data, max_score=200
# Warm start from 2R epoch 3
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.1 --rank-weight 1.0 --rank-margin-target 10 \
    --endgame-fraction 0.5 --endgame-threshold 100 \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 10 --batch-size 12288 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2s_best.pt \
    --save-dir /content/checkpoints/pillar2s

In [ ]:
# Copy ALL epoch checkpoints to Drive for bracket eval
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2s/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2s_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2s/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2s_{f}')